# V2 Pick-and-Place Training — Horizontal Side Approach

This notebook replaces the top-down Cartesian approach from v1 with the **elbow-back, horizontal side approach** used in `scripts/demo_control_panel.py`. The gripper enters the workspace horizontally (level with the object) rather than descending from above, so the gripper opening naturally faces the target without requiring forearm-yaw tricks.

### Key motion pattern
1. Arm starts in the **GUARD_R** retracted pose (shoulder pitched *back* behind the torso, forearm folded up)
2. The shoulder pitches **forward** in a wide arc that keeps the hand clear of the tabletop throughout the swing
3. The arm extends horizontally at object height; the elbow folds slightly toward the torso to drop the gripper in front of the object
4. A short forward push slides the object into the open gripper

### Each cell prints
- Gripper pad world position `(x, y, z)`
- Live object positions from `/tmp/reachy_scene_overrides.json` (physics backend) or scene-YAML defaults (kinematic)
- `[FELL]` warning when an object's z drops below the table surface

> **Run inside the simulator container** (mujoco-remote backend preferred for physics feedback):
> ```bash
> REACHY_SIM_SCENE=scenes/tabletop_demo.yaml ./scripts/start_sim.sh
> docker compose exec reachy-sim jupyter notebook --ip=0.0.0.0 --no-browser
> ```

## Cell 1 — Imports & helpers

Sets up the Python path (works in both Jupyter and `exec` contexts), imports the SDK and motion modules, and defines `print_state()` — the status reporter called at the end of every motion cell.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import pathlib
import sys
import time
import uuid

# ── Path setup ────────────────────────────────────────────────────────────────
# Locate the repo src/ regardless of whether this runs in Jupyter or via exec.
def _find_src() -> pathlib.Path:
    for candidate in [
        pathlib.Path.cwd() / "src",
        pathlib.Path.cwd().parent / "src",
        pathlib.Path("/opt/src"),
        pathlib.Path("/tmp/reachy-tabletop-ai/src"),
    ]:
        if (candidate / "reachy_ai").is_dir():
            return candidate
    raise RuntimeError("Cannot find src/reachy_ai — run from the repo root or inside the container.")

_SRC = _find_src()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from reachy_ai.motion import primitives as P
from reachy_ai.motion.kinematics import CartesianPlanner, R_ARM_JOINTS
from reachy_ai.motion.safety import gate_check
from reachy_ai.scene.awareness import SceneModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("v2_nb")

# ── Scene constants ────────────────────────────────────────────────────────────
# Initial positions from scenes/tabletop_demo.yaml (used as fallback in kinematic mode).
_SCENE_INIT = {
    "red_cube":      [0.42, -0.18, 0.77],
    "blue_cylinder": [0.46, -0.08, 0.79],
}
_TABLE_SURFACE_Z = 0.74

# Sentinel files for the between-episode physics reset handshake.
_RESET_REQUEST = pathlib.Path("/tmp/reachy_reset_request")
_RESET_ACK     = pathlib.Path("/tmp/reachy_reset_ack")

# ── Elbow-back GUARD_R pose (from scripts/demo_control_panel.py) ──────────────
# +pitch swings the upper arm BACK behind the torso (relay-runner elbow-jab
# posture).  The tight elbow fold (-115°) keeps the forearm high.  The arm
# then swings FORWARD by pitching the shoulder negative to reach the workspace,
# clearing the tabletop with the hand during the whole arc.
GUARD_R = {
    "r_shoulder_pitch":  30.0,    # upper arm swings BACK behind torso
    "r_shoulder_roll":  -10.0,
    "r_arm_yaw":          0.0,
    "r_elbow_pitch":   -115.0,    # forearm folded tight toward shoulder
    "r_forearm_yaw":      0.0,
    "r_wrist_pitch":     15.0,
    "r_wrist_roll":       0.0,
    "r_gripper":        -45.0,    # open  (neg = open for right gripper)
}

# ── Status reporter ────────────────────────────────────────────────────────────

def _read_obj_positions() -> dict:
    """Return {object_id: [x, y, z]} from the physics overrides file, or scene defaults."""
    p = pathlib.Path("/tmp/reachy_scene_overrides.json")
    try:
        if p.exists():
            data = json.loads(p.read_text())
            if data:
                return data
    except (json.JSONDecodeError, OSError):
        pass
    return _SCENE_INIT  # kinematic mode — objects don't move


def print_state(arm, planner: CartesianPlanner, label: str = "") -> None:
    """Print gripper pad world position and each tracked object's position."""
    q = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
    gx, gy, gz = planner.fk_world(q)
    tag = f"[{label}] " if label else ""
    print(f"{tag}Gripper pad: ({gx:.3f}, {gy:.3f}, {gz:.3f})")
    objs = _read_obj_positions()
    for oid, pos in objs.items():
        status = "FELL" if pos[2] < _TABLE_SURFACE_Z - 0.03 else "OK"
        print(f"  {oid}: ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) [{status}]")


def reset_scene(timeout: float = 8.0) -> bool:
    """Request a physics reset and wait for the server's ack (file-sentinel handshake)."""
    gen = uuid.uuid4().hex
    _RESET_ACK.unlink(missing_ok=True)
    _RESET_REQUEST.write_text(gen)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _RESET_ACK.exists() and _RESET_ACK.read_text().strip() == gen:
            log.info("Scene reset confirmed.")
            return True
        time.sleep(0.1)
    log.warning("reset_scene: timed out waiting for ack")
    return False

print("Imports OK — src found at:", _SRC)

## Step 1 — Initialize robot and pick-and-place environment

Connects to the Reachy SDK server, checks the safety gate, loads the scene model, and creates the Cartesian planner. Prints initial gripper and object positions.

In [ ]:
from reachy_sdk import ReachySDK

_HOST = os.environ.get("REACHY_IP", "localhost")
_PORT = int(os.environ.get("REACHY_SDK_PORT", 50051))

if not gate_check():
    raise RuntimeError("Safety gate failed — set REACHY_ENABLE_MOTION=true on physical robot with operator present.")

# Load scene model (provides object geometry and collision boundaries).
_SCENE_YAML = next(
    p for p in [
        pathlib.Path("/opt/scenes/tabletop_demo.yaml"),
        _SRC.parent / "scenes" / "tabletop_demo.yaml",
    ]
    if p.exists()
)
scene = SceneModel.from_yaml(str(_SCENE_YAML))
log.info("Scene loaded: %s  table_surface_z=%.3f", _SCENE_YAML.name, scene.table_surface_z)

robot = ReachySDK(host=_HOST, sdk_port=_PORT)
time.sleep(0.8)
arm = robot.r_arm
if arm is None:
    raise RuntimeError("Right arm not found — is the simulator running?")

planner = CartesianPlanner(arm, scene=scene, side="right")

print("Robot connected — right arm ready")
print(f"Object initial positions (from scene YAML):")
for oid, pos in _SCENE_INIT.items():
    print(f"  {oid}: {pos}")
print_state(arm, planner, label="Step 1 init")

## Step 2 — Head looks at table objects

Turns on the head and steers the neck so both cameras center on the target object. The look-at point is the red cube's world position. The head will track the object again in Step 8 as it is lifted.

In [ ]:
# Aim the head at the centroid of the manipulable objects so both are in frame.
_obj_positions = _read_obj_positions()
_xs = [v[0] for v in _obj_positions.values()]
_ys = [v[1] for v in _obj_positions.values()]
_zs = [v[2] for v in _obj_positions.values()]
_look_target = (sum(_xs) / len(_xs), sum(_ys) / len(_ys), sum(_zs) / len(_zs))

P.look_at(robot, _look_target, duration=1.2)
time.sleep(0.3)

print(f"Head looking at: ({_look_target[0]:.3f}, {_look_target[1]:.3f}, {_look_target[2]:.3f})")
print_state(arm, planner, label="Step 2 look-at")

## Step 3 — Right arm ON in home position

Activates the right arm motors and moves all joints to the zero (HOME) pose. The arm hangs straight down at the robot's side.

In [ ]:
robot.turn_on("r_arm")
time.sleep(0.3)

P.smooth_move(arm, P.HOME, duration=1.5)
P.wait_until(arm, P.HOME, tol=8.0, timeout=3.0)
time.sleep(0.3)

print("Right arm ON and at HOME")
print_state(arm, planner, label="Step 3 home")

## Step 4 — Raise arm using the elbow-back GUARD_R motion

This is the key departure from v1.  Instead of lifting the arm straight forward (which sweeps the gripper into the tabletop), we first **pitch the shoulder backward** (positive pitch) so the upper arm and elbow swing *behind* the torso — the same retracted posture Reachy uses in the control-panel demo.  From this guard position the shoulder can then pitch *forward* and the elbow/upper arm clear the table edge throughout the full arc.

```
GUARD_R  r_shoulder_pitch = +30°   ← arm swings BACK behind torso
         r_elbow_pitch    = -115°  ← forearm folded tight upward
```

In [ ]:
# Swing arm into the elbow-back GUARD_R retracted pose.
P.smooth_move(arm, GUARD_R, duration=1.5)
P.wait_until(arm, GUARD_R, tol=8.0, timeout=3.0, reassert=True)
time.sleep(0.4)

print("Arm in GUARD_R — elbow back, forearm folded, gripper open")
print_state(arm, planner, label="Step 4 GUARD_R")

## Step 5a — Swing arm above table (GUARD_R → IK to above-object)

From GUARD_R the arm swings forward to a point **directly above the red cube** at safe clearance height.  The Cartesian IK planner computes the joint angles (as the control-panel demo does internally with `reach()`), so the arm finds the right configuration regardless of physics tracking lag.

Target: `(0.42, -0.18, 0.95)` — cube x/y, 18 cm above table.

In [ ]:
from reachy_ai.motion.kinematics import UnreachableError

# IK-solve from current (GUARD_R) joint angles to a point above the object.
# The arm swings forward as one smooth motion — the IK handles the joint angles.
_TARGET_ABOVE = (0.42, -0.18, 0.95)
_seed = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
try:
    _q_above = planner.solve(_TARGET_ABOVE, seed=_seed)
except UnreachableError:
    raise RuntimeError(f"IK could not reach {_TARGET_ABOVE} — check scene geometry.")

ABOVE_OBJ = dict(zip(R_ARM_JOINTS, _q_above))
ABOVE_OBJ["r_gripper"] = -45.0  # keep gripper open

P.smooth_move(arm, ABOVE_OBJ, duration=2.5)
P.wait_until(arm, ABOVE_OBJ, tol=8.0, timeout=5.0, reassert=True)
time.sleep(0.5)

print("Arm above object — IK joint angles found")
print_state(arm, planner, label="Step 5a above-obj")

## Step 5b — Lower gripper in front of object (elbow moves backward)

From above the object, IK-solve to a point **in front of the cube** at object height.  Pulling x back to 0.28 m (14 cm behind the cube's face) naturally moves the elbow backward in world space as the arm un-reaches slightly.  The gripper descends to the object's z level ready for the horizontal side approach.

In [ ]:
# IK to a point in FRONT of the cube at object height.
# x=0.28: 14 cm behind the cube face (cube at x=0.42, half-width=0.03 → face at x=0.39)
# z=0.80: 3 cm above table surface, at cube centre height.
_TARGET_FRONT = (0.28, -0.18, 0.80)
_seed2 = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
try:
    _q_front = planner.solve(_TARGET_FRONT, seed=_seed2)
except UnreachableError:
    raise RuntimeError(f"IK could not reach {_TARGET_FRONT}.")

FRONT_OF_OBJ = dict(zip(R_ARM_JOINTS, _q_front))
FRONT_OF_OBJ["r_gripper"] = -45.0  # open

P.smooth_move(arm, FRONT_OF_OBJ, duration=2.0)
P.wait_until(arm, FRONT_OF_OBJ, tol=8.0, timeout=4.0, reassert=True)
time.sleep(0.5)

print("Gripper lowered in front of object — elbow has moved back")
print_state(arm, planner, label="Step 5b front-of-obj")
print(f"  [Target: {_TARGET_FRONT}  cube: {_SCENE_INIT['red_cube']}]")

## Step 6 — Slide gripper forward to nestle object inside

With the gripper horizontal and level with the cube, we extend the arm slightly forward by pitching the shoulder a few more degrees negative.  The open gripper slides around the cube so the cube sits inside the pads.

In [ ]:
# IK to the cube's position at object height — gripper slides forward to capture.
_TARGET_NESTLE = (0.42, -0.18, 0.80)
_seed3 = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
try:
    _q_nestle = planner.solve(_TARGET_NESTLE, seed=_seed3)
except UnreachableError:
    raise RuntimeError(f"IK could not reach {_TARGET_NESTLE}.")

NESTLE = dict(zip(R_ARM_JOINTS, _q_nestle))
NESTLE["r_gripper"] = -45.0  # still open

P.smooth_move(arm, NESTLE, duration=1.2)
P.wait_until(arm, NESTLE, tol=8.0, timeout=3.0, reassert=True)
time.sleep(0.5)

print("Gripper advanced to object — nestled in pads")
print_state(arm, planner, label="Step 6 nestle")

## Step 7 — Close gripper around the object

Closes the right gripper smoothly.  `P.close_gripper` moves `r_gripper` from -45° (open) to -5° (firm hold).

In [ ]:
P.close_gripper(arm, duration=0.8)
time.sleep(1.0)  # settle before lifting

print("Gripper closed")
print_state(arm, planner, label="Step 7 grip")

## Step 8 — Lift object to head height; head and stereo view follow

The arm pitches upward, raising the gripped object to approximately Reachy's head height (~1.15 m).  The head (`look_at`) tracks the gripper pad world position so the stereo cameras stay on the object throughout the lift.

> Object z should rise above 1.0 m — if it stays at 0.77 the grasp missed; re-run from Step 3.

In [ ]:
# Lift: pitch shoulder steeply forward-upward (-80°) while keeping the elbow
# partially bent (-80°) so the object clears any table-edge clutter.
LIFT_HIGH = {
    "r_shoulder_pitch": -80.0,   # arm steeply forward — raises gripper to head level
    "r_shoulder_roll":   5.0,
    "r_arm_yaw":         0.0,
    "r_elbow_pitch":    -80.0,
    "r_forearm_yaw":     0.0,
    "r_wrist_pitch":     0.0,    # keep gripper horizontal so object doesn't tip
    "r_wrist_roll":      0.0,
    "r_gripper":        -5.0,    # closed
}

# Lift slowly while updating the head gaze every 0.5 s so stereo follows.
_LIFT_STEPS = 6
_LIFT_DUR   = 2.5
_start_pose = {n: getattr(arm, n).present_position for n in LIFT_HIGH}
_dt = _LIFT_DUR / _LIFT_STEPS
for _step in range(1, _LIFT_STEPS + 1):
    _t = _step / _LIFT_STEPS
    _interp = {n: _start_pose[n] + _t * (LIFT_HIGH[n] - _start_pose[n]) for n in LIFT_HIGH}
    for _n, _v in _interp.items():
        getattr(arm, _n).goal_position = _v
    # Update head gaze toward the current gripper position.
    _q = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
    _gp = planner.fk_world(_q)
    P.look_at(robot, _gp, duration=_dt * 0.9)
    time.sleep(_dt)

P.wait_until(arm, LIFT_HIGH, tol=10.0, timeout=3.0, reassert=True)
time.sleep(0.5)

print("Object lifted to head height — stereo cameras tracking")
print_state(arm, planner, label="Step 8 lift")

## Step 9 — Place object to the left

Swings the arm toward positive-y (Reachy's left / +y side of the table) using `r_shoulder_roll` and `r_arm_yaw`, then lowers the object to the table surface at approximately (0.42, +0.15, 0.77) and releases.

In [ ]:
# Swing left: shoulder_roll toward +y, arm_yaw to rotate the whole arm.
SWING_LEFT_HIGH = {
    "r_shoulder_pitch": -60.0,
    "r_shoulder_roll":  -15.0,   # negative roll swings arm toward +y (robot's left)
    "r_arm_yaw":        -25.0,   # yaw rotates upper arm counterclockwise (also toward +y)
    "r_elbow_pitch":    -70.0,
    "r_forearm_yaw":     0.0,
    "r_wrist_pitch":    10.0,
    "r_wrist_roll":      0.0,
    "r_gripper":        -5.0,    # closed — still carrying
}

P.look_at(robot, (0.42, 0.15, 0.77), duration=0.8)
P.smooth_move(arm, SWING_LEFT_HIGH, duration=1.8)
P.wait_until(arm, SWING_LEFT_HIGH, tol=10.0, timeout=3.0, reassert=True)
time.sleep(0.3)

print_state(arm, planner, label="Step 9 over-place")

# Lower to place height.
PLACE_LEFT = {
    **SWING_LEFT_HIGH,
    "r_shoulder_pitch": -55.0,   # slightly less steep → arm lowers
    "r_elbow_pitch":    -80.0,   # more bend → hand descends
    "r_wrist_pitch":     0.0,
}

P.smooth_move(arm, PLACE_LEFT, duration=1.2)
P.wait_until(arm, PLACE_LEFT, tol=10.0, timeout=2.5, reassert=True)
time.sleep(0.5)

# Release.
P.open_gripper(arm, duration=0.8)
time.sleep(0.8)

print("Object placed to the left — gripper released")
print_state(arm, planner, label="Step 9 placed")

## Step 10 — Raise arm and return to side

Retracts the arm via the `stow_from_side` primitive (jumping-jack reversal), which:
1. Abducts the shoulder roll out to the side
2. Straightens the elbow with the arm held out
3. Lowers the shoulder roll so the arm swings down by Reachy's side
4. Settles at HOME and turns off motors

This route never crosses the tabletop.

In [ ]:
# Head returns to a neutral forward-looking pose.
P.look_at(robot, (0.50, 0.0, 0.90), duration=0.8)

# Stow the arm — routes outward via the jumping-jack abduction path.
P.stow_from_side(robot, arm, duration=3.0)
time.sleep(0.5)

print("Arm stowed — motors off")
# Arm motors are off; FK of last commanded position.
print_state(arm, planner, label="Step 10 stowed")

---
## Training loop

Wraps the full sequence in a loop.  Between episodes the scene is reset via the file-sentinel handshake so the physics server respawns the cube and cylinder at their starting positions.

Adjust `N_EPISODES` to taste.

In [ ]:
N_EPISODES = 3

_T_ABOVE  = (0.42, -0.18, 0.95)   # above red cube
_T_FRONT  = (0.28, -0.18, 0.80)   # in front at object height
_T_NESTLE = (0.42, -0.18, 0.80)   # at object position


def _ik_move(target, seed_arm, duration: float = 2.0, label: str = "") -> dict:
    """IK-solve to target and smooth_move there.  Returns the joint pose dict."""
    seed = [getattr(seed_arm, n).present_position for n in R_ARM_JOINTS]
    try:
        q = planner.solve(target, seed=seed)
    except UnreachableError:
        raise RuntimeError(f"IK unreachable: {target}")
    pose = dict(zip(R_ARM_JOINTS, q))
    pose["r_gripper"] = getattr(seed_arm, "r_gripper").present_position
    P.smooth_move(seed_arm, pose, duration=duration)
    P.wait_until(seed_arm, pose, tol=8.0, timeout=duration + 2.0, reassert=True)
    time.sleep(0.3)
    if label:
        print_state(seed_arm, planner, label=label)
    return pose


def _settle(pose: dict, dur: float = 1.5, tol: float = 8.0, timeout: float = 3.0) -> None:
    P.smooth_move(arm, pose, duration=dur)
    P.wait_until(arm, pose, tol=tol, timeout=timeout, reassert=True)
    time.sleep(0.3)


def run_episode(ep: int) -> bool:
    log.info("=== Episode %d / %d ===", ep, N_EPISODES)

    # Step 2: look at objects
    objs = _read_obj_positions()
    xs = [v[0] for v in objs.values()]
    ys = [v[1] for v in objs.values()]
    zs = [v[2] for v in objs.values()]
    P.look_at(robot, (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs)), duration=1.0)

    # Step 3: arm ON + home
    robot.turn_on("r_arm")
    time.sleep(0.3)
    _settle(P.HOME, dur=1.5)
    print_state(arm, planner, label=f"ep{ep} home")

    # Step 4: GUARD_R (elbow-back bird-wing)
    _settle(GUARD_R, dur=1.5, tol=8.0)
    print_state(arm, planner, label=f"ep{ep} GUARD_R")

    # Step 5a: IK swing above object
    _ik_move(_T_ABOVE, arm, duration=2.5, label=f"ep{ep} above-obj")

    # Step 5b: IK lower in front of object
    _ik_move(_T_FRONT, arm, duration=2.0, label=f"ep{ep} front-of-obj")

    # Step 6: IK nestle at object
    nestle_pose = _ik_move(_T_NESTLE, arm, duration=1.2, label=f"ep{ep} nestle")

    # Step 7: close gripper
    P.close_gripper(arm, duration=0.8)
    time.sleep(1.0)
    print_state(arm, planner, label=f"ep{ep} grip")

    # Step 8: lift to head height, head tracks
    _start = {n: getattr(arm, n).present_position for n in LIFT_HIGH}
    _dt = 2.5 / 6
    for _s in range(1, 7):
        _t = _s / 6
        _ip = {n: _start[n] + _t * (LIFT_HIGH[n] - _start[n]) for n in LIFT_HIGH}
        for _n, _v in _ip.items():
            getattr(arm, _n).goal_position = _v
        _gp = planner.fk_world([getattr(arm, n).present_position for n in R_ARM_JOINTS])
        P.look_at(robot, _gp, duration=_dt * 0.9)
        time.sleep(_dt)
    P.wait_until(arm, LIFT_HIGH, tol=10.0, timeout=3.0, reassert=True)
    time.sleep(0.5)
    print_state(arm, planner, label=f"ep{ep} lift")

    # Step 9: place to the left
    P.look_at(robot, (0.42, 0.15, 0.77), duration=0.8)
    _settle(SWING_LEFT_HIGH, dur=1.8, tol=10.0)
    _settle(PLACE_LEFT, dur=1.2, tol=10.0)
    P.open_gripper(arm, duration=0.8)
    time.sleep(0.8)
    print_state(arm, planner, label=f"ep{ep} placed")

    # Step 10: stow arm
    P.look_at(robot, (0.50, 0.0, 0.90), duration=0.8)
    P.stow_from_side(robot, arm, duration=3.0)
    print_state(arm, planner, label=f"ep{ep} stowed")

    final = _read_obj_positions().get("red_cube", _SCENE_INIT["red_cube"])
    success = final[1] > 0.0
    log.info("Episode %d result: %s  (red_cube y=%.3f)", ep, "SUCCESS" if success else "MISS", final[1])
    return success


successes = 0
for ep_num in range(1, N_EPISODES + 1):
    if ep_num > 1:
        log.info("Resetting scene for episode %d...", ep_num)
        ok = reset_scene(timeout=8.0)
        if not ok:
            log.warning("Scene reset timed out — running anyway")
        time.sleep(1.0)
    try:
        if run_episode(ep_num):
            successes += 1
    except Exception as exc:
        log.error("Episode %d failed with exception: %s", ep_num, exc)

print(f"\n=== Training complete: {successes}/{N_EPISODES} episodes succeeded ===")